# Part 4 — Python + SQL Integration

Implements the command-line reporting tool required by the assignment.

The tool:
1. Takes report type: daily / weekly / monthly
2. Takes a date range
3. Connects to SQLite
4. Shows total orders, revenue, unique customers
5. Shows top 3 products
6. Compares the selected period with the previous period

The reporting logic uses Python's built-in `sqlite3` module for the database connection.

In [1]:
import sqlite3
from datetime import datetime, timedelta

DB_FILE = "ecommerce.db"

def connect_database():
    return sqlite3.connect(DB_FILE)

## 1. Validate Dates

In [2]:
def parse_date(date_text):
    try:
        return datetime.strptime(date_text, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            "Date must be in YYYY-MM-DD format."
        )

## 2. Calculate Previous Period

In [3]:
def previous_period(start_date, end_date):
    days = (end_date - start_date).days + 1

    previous_end = start_date - timedelta(days=1)
    previous_start = previous_end - timedelta(days=days - 1)

    return previous_start, previous_end

## 3. Generate Report

In [4]:
def generate_summary_report(report_type, start_date, end_date):

    report_type = report_type.lower()

    if report_type not in ["daily", "weekly", "monthly"]:
        raise ValueError(
            "Report type must be daily, weekly, or monthly."
        )

    start_date = parse_date(start_date)
    end_date = parse_date(end_date)

    if start_date > end_date:
        raise ValueError(
            "Start date cannot be after end date."
        )

    previous_start, previous_end = previous_period(
        start_date,
        end_date
    )

    conn = connect_database()
    cursor = conn.cursor()

    start = start_date.isoformat()
    end = end_date.isoformat()
    prev_start = previous_start.isoformat()
    prev_end = previous_end.isoformat()

    # Summary for selected period
    summary_query = '''
    SELECT
        COUNT(DISTINCT o.order_id) AS total_orders,
        COALESCE(SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ), 0) AS revenue,
        COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM orders o
    LEFT JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE date(o.order_date) BETWEEN ? AND ?
      AND o.status NOT IN ('CANCELLED');
    '''

    cursor.execute(summary_query, (start, end))
    total_orders, revenue, unique_customers = cursor.fetchone()

    # Previous period summary
    cursor.execute(summary_query, (prev_start, prev_end))
    prev_orders, prev_revenue, prev_customers = cursor.fetchone()

    # Top 3 products
    top_products_query = '''
    SELECT
        p.product_name,
        ROUND(SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ), 2) AS revenue
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    WHERE date(o.order_date) BETWEEN ? AND ?
      AND o.status NOT IN ('CANCELLED')
    GROUP BY p.product_id, p.product_name
    ORDER BY revenue DESC
    LIMIT 3;
    '''

    cursor.execute(top_products_query, (start, end))
    top_products = cursor.fetchall()

    def percentage_change(current, previous):
        if previous in (None, 0):
            return None
        return ((current - previous) / previous) * 100

    order_change = percentage_change(
        total_orders or 0,
        prev_orders or 0
    )

    revenue_change = percentage_change(
        revenue or 0,
        prev_revenue or 0
    )

    customer_change = percentage_change(
        unique_customers or 0,
        prev_customers or 0
    )

    print("=" * 60)
    print("E-COMMERCE SUMMARY REPORT")
    print("=" * 60)

    print(f"Report Type       : {report_type.title()}")
    print(f"Date Range        : {start} to {end}")
    print(f"Previous Period   : {prev_start} to {prev_end}")

    print("\nCurrent Period")
    print(f"Total Orders      : {total_orders}")
    print(f"Revenue           : {revenue:.2f}")
    print(f"Unique Customers  : {unique_customers}")

    print("\nPrevious Period")
    print(f"Total Orders      : {prev_orders}")
    print(f"Revenue           : {prev_revenue:.2f}")
    print(f"Unique Customers  : {prev_customers}")

    print("\nPercentage Change")
    print(
        f"Orders            : "
        f"{order_change:.2f}%"
        if order_change is not None
        else "Orders            : N/A"
    )
    print(
        f"Revenue           : "
        f"{revenue_change:.2f}%"
        if revenue_change is not None
        else "Revenue           : N/A"
    )
    print(
        f"Customers         : "
        f"{customer_change:.2f}%"
        if customer_change is not None
        else "Customers         : N/A"
    )

    print("\nTop 3 Products")

    if top_products:
        for i, (product, product_revenue) in enumerate(
            top_products,
            start=1
        ):
            print(
                f"{i}. {product} - "
                f"{product_revenue:.2f}"
            )
    else:
        print("No products found.")

    conn.close()

## 4. Example Report Runs

In [5]:
# Example:
# Change these dates if your generated dataset has a different range.

generate_summary_report(
    "monthly",
    "2025-05-01",
    "2025-05-31"
)

E-COMMERCE SUMMARY REPORT
Report Type       : Monthly
Date Range        : 2025-05-01 to 2025-05-31
Previous Period   : 2025-03-31 to 2025-04-30

Current Period
Total Orders      : 52
Revenue           : 6108831.32
Unique Customers  : 51

Previous Period
Total Orders      : 43
Revenue           : 3642861.86
Unique Customers  : 40

Percentage Change
Orders            : 20.93%
Revenue           : 67.69%
Customers         : 27.50%

Top 3 Products
1. Furniture Product 51 - 348008.98
2. T-Shirts Product 48 - 246467.99
3. Camera Product 22 - 235337.42


## 5. Interactive Command-Line Version

In [6]:
def run_cli():

    print("=" * 60)
    print("E-COMMERCE ORDER REPORTING TOOL")
    print("=" * 60)

    report_type = input(
        "Enter report type (daily/weekly/monthly): "
    ).strip().lower()

    start_date = input(
        "Enter start date (YYYY-MM-DD): "
    ).strip()

    end_date = input(
        "Enter end date (YYYY-MM-DD): "
    ).strip()

    try:
        generate_summary_report(
            report_type,
            start_date,
            end_date
        )
    except Exception as e:
        print(f"Error: {e}")

# Uncomment this line when you want interactive input:
# run_cli()